# Compare trained AE models

Load all `.pt` checkpoints from a results directory, reconstruct each model
from saved `model_config`, run evaluation on the test set, and display all
figures and metrics side by side.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from model.dl import AutoEncoder
from eval_ae import evaluate

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

In [ ]:
# ============================================================
# CONFIGURE: point this at the directory containing .pt files
# ============================================================
RESULTS_DIR = "results/ae_states"

# Find all checkpoint files
results_path = Path(RESULTS_DIR)
pt_files = sorted(results_path.glob("model_*.pt"))
print(f"Found {len(pt_files)} checkpoints in {RESULTS_DIR}:")
for f in pt_files:
    print(f"  {f.name}")

In [ ]:
# ============================================================
# Load data and prepare test set (same split as training)
# ============================================================
STATE_COLS = ["RAS_s", "RAF_s", "MEK_s", "NFB_s", "ERK_s"]

df = pd.read_parquet("synthetic_EGFR_data.parquet")
states = np.stack([np.concatenate(df[c].values) for c in STATE_COLS], axis=1).astype(np.float32)

traj_ids = np.arange(len(df))
tr_ids, te_ids = train_test_split(traj_ids, test_size=0.2, random_state=42)

traj_len = len(df[STATE_COLS[0]].iloc[0])
test_idx = np.concatenate([np.arange(i * traj_len, (i + 1) * traj_len) for i in te_ids])
test_states = states[test_idx]
test_traj_lengths = np.full(len(te_ids), traj_len)
test_generator_labels = np.repeat(df["generator"].values[te_ids], traj_len)
state_names = [c.replace("_s", "") for c in STATE_COLS]

print(f"Test set: {len(test_states)} samples, {len(te_ids)} trajectories")

In [ ]:
# ============================================================
# Load each checkpoint, reconstruct model, run evaluation
# ============================================================
results = {}

for pt_file in pt_files:
    ckpt = torch.load(pt_file, map_location=device, weights_only=False)

    # Reconstruct model from saved config
    if "model_config" in ckpt:
        cfg = ckpt["model_config"]
        model = AutoEncoder(
            input_dim=cfg["input_dim"],
            hidden_dims=cfg["hidden_dims"],
            latent_dim=cfg["latent_dim"],
        ).to(device)
    else:
        # Fallback: infer from state_dict
        sd = ckpt["model_state_dict"]
        input_dim = sd["encoder.net.0.weight"].shape[1]
        latent_dim = [v for k, v in sd.items() if "encoder" in k and "weight" in k][-1].shape[0]
        print(f"  {pt_file.name}: no model_config, inferred input_dim={input_dim}, latent_dim={latent_dim}")
        # Try common hidden_dims; this may need adjustment
        hidden_dims = tuple(
            sd[k].shape[0] for k in sd if "encoder" in k and "weight" in k
        )[:-1]
        model = AutoEncoder(
            input_dim=input_dim,
            hidden_dims=hidden_dims,
            latent_dim=latent_dim,
        ).to(device)

    model.load_state_dict(ckpt["model_state_dict"])

    # Build a label from config + timestamp
    ts = ckpt.get("train_start", pt_file.stem)
    if "model_config" in ckpt:
        cfg = ckpt["model_config"]
        label = f"H{cfg['hidden_dims']}_L{cfg['latent_dim']}  ({ts})"
    else:
        label = pt_file.stem

    result = evaluate(
        model=model,
        states=test_states,
        traj_lengths=test_traj_lengths,
        state_names=state_names,
        name=label,
        generator_labels=test_generator_labels,
    )
    results[label] = result
    print(f"Evaluated: {label}  (MSE={result.metrics['mse_overall']:.6f})")

print(f"\nLoaded {len(results)} models.")

---
## Summary table

In [ ]:
rows = []
for label, r in results.items():
    m = r.metrics
    row = {
        "model": label,
        "latent_dim": m["latent_dim"],
        "MSE_overall": m["mse_overall"],
        "vel_corr": m.get("velocity_correlation", None),
    }
    # Per-state MSE
    for name, val in zip(m["state_names"], m["mse_per_state"]):
        row[f"MSE_{name}"] = val
    # k-NN overlap
    for k, v in m.get("knn_overlap", {}).items():
        row[f"kNN_k{k}"] = v
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index("model")
summary_df.style.format("{:.4f}").highlight_min(
    subset=[c for c in summary_df.columns if "MSE" in c], color="#d4edda"
).highlight_max(
    subset=[c for c in summary_df.columns if c in ("vel_corr", "kNN_k10", "kNN_k50")], color="#d4edda"
)

---
## Training curves

In [ ]:
# Load training history from each checkpoint and plot
skip = 5
n_models = len(pt_files)
fig, axes = plt.subplots(n_models, 2, figsize=(12, 3.5 * n_models), squeeze=False)

for i, pt_file in enumerate(pt_files):
    ckpt = torch.load(pt_file, map_location="cpu", weights_only=False)
    history = ckpt.get("history", {})
    train_loss = history.get("train_loss", [])
    val_loss = history.get("val_loss", [])

    if not train_loss:
        axes[i, 0].text(0.5, 0.5, "No history saved", ha="center", va="center",
                        transform=axes[i, 0].transAxes)
        continue

    axes[i, 0].plot(train_loss, label="train")
    axes[i, 0].plot(val_loss, label="val")
    axes[i, 0].set_title(pt_file.stem)
    axes[i, 0].set_xlabel("epoch")
    axes[i, 0].set_ylabel("MSE")
    axes[i, 0].legend(fontsize=8)

    axes[i, 1].plot(range(skip, len(train_loss)), train_loss[skip:], label="train")
    axes[i, 1].plot(range(skip, len(val_loss)), val_loss[skip:], label="val")
    axes[i, 1].set_title(f"{pt_file.stem} (epoch {skip}+)")
    axes[i, 1].set_xlabel("epoch")
    axes[i, 1].set_ylabel("MSE")
    axes[i, 1].legend(fontsize=8)

fig.tight_layout()
plt.show()

---
## Evaluation figures per model

Each model's full set of evaluation plots is shown below.

In [ ]:
for label, r in results.items():
    print("=" * 80)
    r.summary()
    for fig_name, fig in r.figures.items():
        # Re-display each figure with a title
        print(f"--- {fig_name} ---")
        display(fig)